# Fraud Detection — Exploratory Data Analysis

## Objective
The purpose of this project is to build a model that can detect fraudulent activity based on transaction data, and answer the following questions: 
    1. Why is `TRANSFER` riskier than `CASH_OUT`, despite being the only two fraud-bearing types?
    2. Is there a `TRANSFER` -> `CASH_OUT` behavioral chain — does fraud money move as `TRANSFER` then `CASH_OUT`?

In this notebook, the objective is to explore the structure and distributions in paysim.csv to (1) build an evidence-based hypothesis for why `TRANSFER` carries more fraud risk than `CASH_OUT`, and (2) confirm if there is a behavioral chain between `TRANSFER` and `CASH_OUT`. 

## Inputs
- `..\data\raw\paysim.csv`
- PaySim dataset — 6,362,620 transaction events, 11 columns
- Column Description:
        - `step`: Time is discretized : 1 step = 1 hour
        - `type`: `CASH_IN`, `CASH_OUT`, `DEBIT`, `PAYMENT`, `TRANSFER`
        - `amount`
        - 

## Outputs
- feature engineering steps
- chain-indicator hypothesis to test in feature engineering

## 1.1 Setup & Imports

Importing necessary libraries and setting plot styling and paths for reprodicability. 

In [1]:
import warnings
warnings.filterwarnings('ignore') # suppress outputs of warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (10, 6)

RAW_DATA_PATH = '../data/raw/paysim.csv'

## 1.2 Load Dataset

Using pandas to read the dataset and verifying the `.csv` has been loaded properly using `.shape` and `.head()`

In [2]:
df = pd.read_csv(RAW_DATA_PATH)
print(df.shape)
df.head()

(6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


## 1.3 Sanity Checks

Checking for: incorrect data types, missing values, duplicates, any impossible or suspicious values, and unexpected categories. These checks ensure analysis is done without inaccurate or missing data, so that down stream models can be trained and tested on good data

In [7]:
df.dtypes

step                int64
type                  str
amount            float64
nameOrig              str
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest              str
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object

All data types look to be correct — continuous features like `amount` and `oldbalanceOrg` are `float64`, binary features are `int64`, and names are `str`. 

In [8]:
df.isnull().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

all sums are 0 therefore no obvious missing values. 

Given the dataset was produced from simulated transaction there is a high potential for duplicated rows to exist, so `.duplicated().sum()` will give sums of any duplicate transaction.

In [10]:
df.duplicated().sum()

np.int64(0)

np.int64(0) means there were no duplicate rows. 

To make sure continuous variables are as they should be with no impossible values. 

In [11]:
df.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,2.433972e+02,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,1.290820e-03,2.514687e-06
std,1.423320e+02,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,3.590480e-02,1.585775e-03
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.560000e+02,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.390000e+02,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,0.000000e+00,0.000000e+00
75%,3.350000e+02,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,0.000000e+00,0.000000e+00
max,7.430000e+02,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00,1.000000e+00


step: large jump from 75% being 3.350000e+02 to max 7.430000e+02 not impossible but i expect to see a right skewed distribution. 
amount: 50% 7.487194e+04 and 75% 2.087215e+05 and max 9.244552e+07 so big jump expect to see right skewed or left skewed? 
oldbalanceOrg: 75% 1.073152e+05 max 5.958504e+07 expect a non normal distribution. also 25% is 0.00
newbalanceOrig: only 75% at 1.442584e+05 25% and 50% are 0.00, but max 4.958504e+07 so expect a high peak on the right side not very balance distribution. 
oldbalanceDest: 75% 9.430367e+05 max 3.560159e+08 so big difference so non normal distribution. is the significant difference in oldbalanceOrg and oldbalanceDest 75% and max and mean an important thing to note? also 25% is 0.00
newbalanceDest: 75% 1.111909e+06 max 3.561793e+08 big dif so expect a non normal distribution and 25% also 0.00 
isFraud and isFlaggedFraud has majority zero in all % categories but no impossible vlaues and this signals that the target `isFraud` is imbalanced. 

to check the three categorical features have any unexpected categories.

In [13]:
df.describe(include='object')

,type,nameOrig,nameDest
count,6362620,6362620,6362620
unique,5,6353307,2722362
top,CASH_OUT,C2098525306,C1286084959
freq,2237500,3,113


## 1.4 Question Reframe

To ensure data holds enough infor to actually answer question "does fraud differ by transaction type".

In [ ]:
df['isFraud'].value_counts()

In [4]:
df.groupby('type')['isFraud'].agg(['sum', 'mean', 'count'])

,sum,mean,count
type,,,
CASH_IN,0,0.000000,1399284
CASH_OUT,4116,0.001840,2237500
DEBIT,0,0.000000,41432
PAYMENT,0,0.000000,2151495
TRANSFER,4097,0.007688,532909


Only two of the different `type` categories — `CASH_OUT` and `TRANSFER` — have cases of fraud. Additionally, although `CASH_OUT` has more cases of fraud (4116) than `TRANSFER` (4097), `TRANSFER` has a significantly higher rate of fraud (0.77%) than `CASH_OUT` (0.18%). This finding caused me to change my initial question of "does fraud differ by transaction types?" to "why does `TRANSFER` have a higher rate of fraud than `CASH_OUT`?".

## 1.5 Target Distribution